In [1]:
%cd ../src

/home/zeke/hello/Booklet/src


In [5]:
import aioaudiobookshelf as abs
import asyncio
import logging
import os
import aiohttp
from aioaudiobookshelf import SessionConfiguration, get_user_client
from aioaudiobookshelf.schema.library import LibraryItemMinifiedBook, LibraryItemMinifiedPodcast
from pathlib import Path
import json

In [ ]:
from src.AudioPlayer import player
from src.AudiobookshelfApiManager import api

In [10]:
config = json.loads(Path('../booklet_config.json').read_text())

In [ ]:
HOST = config['audiobookshelf_url']
KEY = config['audiobookshelf_api_key']

# async def abs_basics():
logger = logging.getLogger()
logger.setLevel(logging.INFO)

async def get_library_id(client):
    for lib in await client.get_all_libraries():
        if lib.name == config['audiobookshelf_library_name']:
            return lib.id_

async def get_titles_from_ids(client, item_ids):
    books = await client.get_library_item_batch_book(item_ids=item_ids)
    return [book.media.metadata.title for book in books]

async def expand_books(client, books):
    return await client.get_library_item_batch_book(item_ids=[b.id_ for b in books])

async def _get_books(client, library_id, filter_id=None):
    books = []
    async for response in client.get_library_items(library_id=library_id, filter_str=filter_id):
        print(f"Fetched {len(response.results)} items from library {library_id}")
        if not response.results:
            break
        books.extend([b for b in response.results if isinstance(b, LibraryItemMinifiedBook)])
    return books

async def get_books(client, library_id, in_progress=False, expanded=False):
    if in_progress:
        filter_str = 'progress.aW4tcHJvZ3Jlc3M%3D'
    else:
        filter_str = None

    books = await _get_books(client, library_id, filter_id=filter_str)
    if expanded:
        return await expand_books(client, books)
    return books


async with aiohttp.ClientSession() as session:
    client = await abs.get_user_client_by_token(
        session_config=SessionConfiguration(
            session=session, url=HOST, logger=logger, pagination_items_per_page=5, token=KEY
        )
    )

    library_id = await get_library_id(client)

    in_progress_filter = 'progress.aW4tcHJvZ3Jlc3M%3D'

    books = await get_books(client, library_id, in_progress=True, expanded=True)
    # display(await get_titles_from_ids(client, [b.id_ for b in books]))
    display([b.media.metadata.title for b in books])
    # book = (await client.get_library_item_batch_book(item_ids=[b.id_ for b in books]))[0]


# asyncio.run(abs_basics())

Fetched 5 items from library 30526570-cbf5-4179-8cbe-90ddf1d84728
Fetched 5 items from library 30526570-cbf5-4179-8cbe-90ddf1d84728
Fetched 5 items from library 30526570-cbf5-4179-8cbe-90ddf1d84728
Fetched 5 items from library 30526570-cbf5-4179-8cbe-90ddf1d84728
Fetched 1 items from library 30526570-cbf5-4179-8cbe-90ddf1d84728
Fetched 0 items from library 30526570-cbf5-4179-8cbe-90ddf1d84728


['Children of Ruin',
 'Shadows of Self',
 'The Salvage Crew',
 'The Hidden Girl and Other Stories',
 'Have Space Suit - Will Travel',
 'Seveneves',
 'Hand of Mars',
 'Algorithms to Live By',
 'An Echo of Things to Come',
 'Project Hail Mary',
 'The Lost Metal',
 'Fourth Wing (Part 2 of 2) (Dramatized Adaptation)',
 'Rhythm of War',
 'Fool Moon',
 'Wind and Truth',
 'Isles of the Emberdark',
 'Successful Reader Middle School Collection C',
 'Exo',
 'Guards! Guards!',
 'Songs of the Dead',
 'The Infinite Extent']